# Imports

In [ ]:
import sys
sys.path.append("..")

In [ ]:
import os
from rouge_score import rouge_scorer
import re
import pymorphy3
import string
import logging

In [ ]:
def log_module(name, level):
    logger = logging.getLogger(name)
    logger.setLevel(level)

    ch = logging.StreamHandler()
    ch.setLevel(level)

    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    ch.setFormatter(formatter)

    logger.addHandler(ch)

In [ ]:
log_module("GraphEvolve", logging.DEBUG)
log_module("AGoTI", logging.WARNING)

In [ ]:
from GraphEvolve.task import SimpleTask
from GraphEvolve.goo import get_default_config
from GraphEvolve.planner import ThreeStepPlanner
from GraphEvolve.solution_database import SolutionDatabase
from GraphEvolve.sampler import PowerLawSampler
from GraphEvolve.mutator import EditMutator, PlanMutator, RandomMutator
from GraphEvolve.runner import Runner

In [ ]:
from AGoTI.model import LLM, ApiVLLMModel

# Models

In [ ]:
generation_config = {
    "temperature": 0.5,
    "top_p": 0.9,
    "max_tokens": 5000,
    "extra_body": {
        "repetition_penalty": 1.0,
        "guided_choice": None,
        "add_generation_prompt": True,
        "guided_regex": None
    }
}

In [ ]:
runner_generation_config = {
    "temperature": 0.1,
    "top_p": 0.9,
    "max_tokens": 3000,
    "extra_body": {
        "repetition_penalty": 1.0,
        "guided_choice": None,
        "add_generation_prompt": True,
        "guided_regex": None,
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
}

In [ ]:
API_KEY = os.getenv("API_KEY")
API_URL = os.getenv("API_URL")
model_name = "Qwen3-235B-A22B-Instruct-2507"
runner_model_name = "tpro-2"
model = ApiVLLMModel(API_KEY, API_URL, model_name, generation_config=generation_config)
runner_model = ApiVLLMModel(API_KEY, API_URL, runner_model_name, generation_config=generation_config)

# Task

In [ ]:
FEEDBACK_PROMPT = """Оцени качество перевода. Исходный текст: "{text}"
Предложенный перевод: "{y_pred}"
Эталонный перевод: "{y_true}"

Напиши краткий фидбек (1-2 предложения), указав на ошибки (если есть) и дав рекомендацию по улучшению. Будь конструктивен."""

In [ ]:
TASK_DESCRIPTION = """Твоя задача перевести предложение с {input} языка на {output} язык. \
Предложение для перевода обозначается строкой {text}. Ответ должен быть в пункте \"translation\""""

In [ ]:
def mean(arr):
    return sum(arr) / max(len(arr), 1)

class CustomTokenizer():
    def __init__(self):
        self.punkt_regex = re.compile('[%s]' % re.escape(string.punctuation))
        self.morph = pymorphy3.MorphAnalyzer()

    def tokenize(self, text):
        return self.preprocess_sentence(text)

    def preprocess_sentence(self, text: str):
        text = text.lower()
        text = self.punkt_regex.sub(' ', text)
        words = [self.morph.parse(token)[0].normal_form for token in text.split() if len(token.strip()) > 0]
        return words

rouge_scorer = rouge_scorer.RougeScorer(['rougeL'], tokenizer=CustomTokenizer())

def rougel(gold, generated):
    return rouge_scorer.score(gold, generated)['rougeL']

In [ ]:
class DaremeruFlores(SimpleTask):
    def __init__(self, model: LLM, original_lang="en"):
        self.model = model
        self.original_lang = original_lang
        self.target_lang = "ru" if original_lang == "en" else "en"

        if self.original_lang == "ru":
            self.description = TASK_DESCRIPTION\
                .replace("{input}", "русского")\
                .replace("{output}", "английский")
        else:
            self.description = TASK_DESCRIPTION\
                .replace("{input}", "английского")\
                .replace("{output}", "русский")

    def dataset_args(self):
        return {"path": "RefalMachine/darumeru", "name": "flores"}

    def train_split(self):
        return "prompt"

    def test_split(self):
        return "test"

    async def evaluate(self, sample, output):
        y_pred = output.get("translation", [""])
        if len(y_pred) == 0:
            return 0
        y_pred = y_pred[0]
        y_true = sample["inputs"][self.target_lang]
        score = rougel(y_true, y_pred).fmeasure
        return score

    async def aggregate(self, scores):
        return mean(scores)

    async def generate_feedback(self, sample, output) -> str:
        y_pred = output.get("translation", [""])
        if len(y_pred) == 0:
            return "Ответ пустой, задача не решена!"
        y_pred = y_pred[0]
        y_true = sample["inputs"][self.target_lang]
        prompt = FEEDBACK_PROMPT\
            .replace("{text}", sample["inputs"][self.original_lang])\
            .replace("{y_pred}", y_pred)\
            .replace("{y_true}", y_true)
        response = await model.generate([{"role": "user", "content": prompt}])
        return response

    def get_goo_kwargs(self, sample):
        return {"replace": {"{text}": sample["inputs"][self.original_lang]}}

In [ ]:
task = DaremeruFlores(model_name)

# GoO config

In [ ]:
goo_config = get_default_config(runner_model)

# Planner

In [ ]:
planner = ThreeStepPlanner(model, goo_config)

# Solution database

In [ ]:
solution_db = SolutionDatabase(goo_config, clear_db=False, db_path="./df_solution_database.db")

In [ ]:
print("\n".join(f"{score:.3f} (id: {id})" for (score, id), _ in solution_db.solutions._data))

# Sampler

In [ ]:
sampler = PowerLawSampler(solution_db)

# Mutator

In [ ]:
edit_mutator = EditMutator(model, goo_config)
plan_mutator = PlanMutator(model, goo_config, planner)
mutator = RandomMutator([edit_mutator, plan_mutator], [0.75, 0.25])

# Basic solution

In [ ]:
basic_plan_draft = """1. Переводит предложение (задается строкой {text}) с английского языка на русский."""

In [ ]:
basic_plan_formal = """\
1. "корень"
"Перевод"
"Переводит предложение (задается строкой {text}) с английского языка на русский. Результат — переведенное предложение."
входы: []
выход: translation

Ответы: {"translation": 1}"""

In [ ]:
basic_goo_json = {'nodes': {'1': {'class': 'корень',
   'parents': [],
   'args': {'prompt': 'Твоя задача перевести текст с английского языка на русский язык. Не давай никаких объяснений и примечаний и сразу начинай перевод.\n\n**Текст:**\n{text}',
    'parsing': 'none',
    'thought_tag': 'translation',
    'name': "Перевод",
    'description': "Переводит предложение (задается строкой {text}) с английского языка на русский. Результат — переведенное предложение."
   }}},
"roots": [1],
"outputs": {"translation": 1}}

# Runner

In [ ]:
runner = Runner(task, solution_db, goo_config, planner, sampler, mutator)

In [ ]:
INIT_WITH_BASIC_SOLUTION = True

In [ ]:
if INIT_WITH_BASIC_SOLUTION:
    basic_solution = await runner.run_solution(basic_plan_draft, basic_plan_formal, basic_goo_json)
    solution_db.add_root(basic_solution)

In [ ]:
await runner.run(1.0, 9)

# Results

In [ ]:
print("\n".join(f"{score:.3f} (id: {id})" for (score, id), _ in solution_db.solutions._data))

## Initial solution

In [ ]:
print(f"Train: {solution_db.root_solution.score}")
root_score = await runner.run_test(solution_db.root_solution)
print(f"Test: {root_score}")

## Best solution

In [ ]:
print(f"Train: {solution_db.best_solution.score}")
best_score = await runner.run_test(solution_db.best_solution)
print(f"Test: {best_score}")

## Basic solution

In [ ]:
if not INIT_WITH_BASIC_SOLUTION:
    basic_solution = await runner.run_solution("", "", basic_goo_json)
    print(f"Train: {basic_solution.score}")

In [ ]:
if not INIT_WITH_BASIC_SOLUTION:
    basic_score = await runner.run_test(basic_solution)
    print(f"Test: {basic_score}")